# Adobe Analytics: daily pull of raw SA360_TrackingID (v84)

Pulls one report per day with the raw tracking ID as the only dimension, then parses
account, market and campaign group in Python.

1. **Config**: fill in the token, client ID and the saved market segment ID.
2. **Pull**: one CSV per day in `aa_v84_raw/`. Safe to re-run; days already saved are skipped.
3. **Parse and aggregate**: writes `aa_daily_market_group.csv` (Day x Market x Brand / Non-brand) and prints the checks.

In [ ]:
# 1. CONFIG (the only cell to edit)

ACCESS_TOKEN = ""
CLIENT_ID    = ""                 # API key from Adobe Developer Console
COMPANY_ID   = "qatara1"
RSID         = "qatarairways-prod"

START_DATE = "2026-05-01"         # first day, inclusive
END_DATE   = "2026-10-05"         # last day, inclusive

DIMENSION = "variables/evar84"    # raw SA360_TrackingID (v84)
METRICS = {                       # output column -> Adobe metric ID
    "flight_search_visits": "metrics/event14",
    "bookings":             "metrics/event27",
    "revenue":              "metrics/event17",   # SalesIncYQ
}
SEGMENTS = [
    "s1952_619e30e8d9c5253ca2de761e",   # BUGs Excluded
    "s1952_6ac61c1d36a75d75a5b13658",          # saved segment: v84 contains any of the market accounts
]

# Country code in the account name (Google-Europe-DE-DE) -> market name
MARKETS = {"DE": "Germany", "CA": "Canada", "SA": "Saudi Arabia",
           "PK": "Pakistan", "MY": "Malaysia", "SG": "Singapore"}

RAW_DIR    = "aa_v84_raw"                   # one CSV per day
OUTPUT_CSV = "aa_daily_market_group.csv"    # final table

# Corporate CA bundle (Zscaler). Ignored if the file does not exist.
CA_BUNDLE = r"C:\Users\EXT13415\notebooks\cacert_merged.pem"

In [ ]:
# 2. PULL: one report per day, raw v84 as the only dimension

import os, csv, time, requests
from datetime import date, timedelta

if CA_BUNDLE and os.path.exists(CA_BUNDLE):
    os.environ["REQUESTS_CA_BUNDLE"] = CA_BUNDLE

assert ACCESS_TOKEN and CLIENT_ID, "Fill in ACCESS_TOKEN and CLIENT_ID in the config cell"
assert not any("PASTE" in s for s in SEGMENTS), "Paste the saved market segment ID in the config cell"

URL = f"https://analytics.adobe.io/api/{COMPANY_ID}/reports"
HEADERS = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "x-api-key": CLIENT_ID,
    "x-proxy-global-company-id": COMPANY_ID,
    "Content-Type": "application/json",
    "Accept": "application/json",
}
TOTALS_CSV = os.path.join(RAW_DIR, "totals.csv")
os.makedirs(RAW_DIR, exist_ok=True)


def post_report(body):
    """POST one report page. Retries rate limits, server errors and timeouts only."""
    for attempt in range(5):
        try:
            print(URL, HEADERS, body)
            r = requests.post(URL, headers=HEADERS, json=body,verify=False)
        except (requests.Timeout, requests.ConnectionError) as e:
            print(repr(e))
            r = None
        if r is not None and r.status_code == 200:
            return r.json()
        if r is not None and r.status_code not in (429, 500, 502, 503, 504):
            raise RuntimeError(f"Adobe API error {r.status_code}: {r.text[:500]}")
        retry_after = r.headers.get("Retry-After", "") if r is not None else ""
        time.sleep(max(int(retry_after) if retry_after.isdigit() else 0, 10 * 2 ** attempt))
    raise RuntimeError("Adobe API still failing after 5 attempts")


def pull_day(day):
    """Return (rows, totals) for one day. Each row is [v84, metric values...]."""
    filters = [{"type": "segment", "segmentId": s} for s in SEGMENTS]
    filters.append({"type": "dateRange",
                    "dateRange": f"{day}T00:00:00.000/{day + timedelta(days=1)}T00:00:00.000"})
    rows, page = [], 0
    while True:
        data = post_report({
            "rsid": RSID,
            "globalFilters": filters,
            "metricContainer": {"metrics": [{"columnId": str(i), "id": m}
                                            for i, m in enumerate(METRICS.values())]},
            "dimension": DIMENSION,
            "settings": {"limit": 50000, "page": page,
                         "countRepeatInstances": True, "nonesBehavior": "return-nones"},
        })
        rows += [[r["value"]] + r["data"] for r in data.get("rows", [])]
        time.sleep(0.6)                      # stay under 120 calls a minute
        if data.get("lastPage", True):
            break
        page += 1
    return rows, data.get("summaryData", {}).get("totals", [])


day, last = date.fromisoformat(START_DATE), date.fromisoformat(END_DATE)
while day <= last:
    path = os.path.join(RAW_DIR, f"v84_{day}.csv")
    if os.path.exists(path):
        print(f"{day}  already saved, skipped")
    else:
        rows, totals = pull_day(day)
        with open(TOTALS_CSV, "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([day] + totals)
        with open(path + ".tmp", "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["v84"] + list(METRICS))
            w.writerows(rows)
        os.replace(path + ".tmp", path)      # the file only exists once the day is complete
        print(f"{day}  {len(rows):>7,} rows")
    day += timedelta(days=1)
print("Pull finished")

In [17]:
# 3. PARSE AND AGGREGATE: Day x Market x Campaign group

import os, glob
import pandas as pd

# account | campaign (may contain pipes) | ad group | keyword | gclid | gclsrc
KEY_RE     = r"^([^|]+)\|(.+)\|([^|]*)\|([^|]*)\|([^|]*)\|([^|]*)$"
ACCOUNT_RE = r"^[^-]+-[^-]+-([A-Z]{2})-[A-Za-z]{2}(?:-Brand)?$"     # Engine-Region-CC-LANG[-Brand]
BRAND_RE   = r"\|Brand\||^_?[A-Z]{2}-Brand[- ]"                      # new format, or legacy US-Brand-Hero-EN_exact
PMAX_RE    = r"perf_max|pmax"
metric_cols = list(METRICS)

daily, checks, other_accounts, brand_check = [], [], [], []
for path in sorted(glob.glob(os.path.join(RAW_DIR, "v84_*.csv"))):
    day = os.path.basename(path)[4:-4]
    df = pd.read_csv(path, dtype={"v84": str}, keep_default_na=False)

    parts = df["v84"].str.extract(KEY_RE)
    df["account"], df["campaign"] = parts[0], parts[1]
    df["market"] = df["account"].str.extract(ACCOUNT_RE)[0].map(MARKETS).fillna("Other")
    df["campaign_group"] = "Non-brand"
    df.loc[df["campaign"].str.contains(BRAND_RE, na=False), "campaign_group"] = "Brand"
    df.loc[df["campaign"].str.contains(PMAX_RE, case=False, na=False), "campaign_group"] = "PMAX"
    df.loc[df["campaign"].isna(), "campaign_group"] = "Unparsed"

    daily.append(df.groupby(["market", "campaign_group"], as_index=False)[metric_cols].sum().assign(day=day))
    other = df[(df["market"] == "Other") & df["account"].notna()]
    other_accounts.append(other.groupby("account", as_index=False)["revenue"].sum())
    brand_check.append(df.assign(brand_account=df["account"].str.endswith("-Brand", na=False))
                         .groupby(["campaign_group", "brand_account"], as_index=False)["revenue"].sum())
    checks.append({
        "day": day,
        "rows": len(df),
        "duplicate_keys": int(df["v84"].duplicated().sum()),
        "revenue_rows": df["revenue"].sum(),
        "revenue_low_traffic": df.loc[df["v84"].str.contains("low-traffic|uniques_exceeded", case=False), "revenue"].sum(),
        "revenue_unparsed": df.loc[df["campaign_group"] == "Unparsed", "revenue"].sum(),
    })

daily = pd.concat(daily, ignore_index=True)

# Final table: every day x market x group, zero where a pulled day has no rows
days = sorted(daily["day"].unique())
grid = pd.MultiIndex.from_product([days, list(MARKETS.values()), ["Brand", "Non-brand"]],
                                  names=["day", "market", "campaign_group"]).to_frame(index=False)
final = grid.merge(daily, how="left", on=["day", "market", "campaign_group"]).fillna(0)
final = final[["day", "market", "campaign_group", "flight_search_visits", "bookings", "revenue"]]
final.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {OUTPUT_CSV}: {len(final):,} rows = {len(days)} days x {len(MARKETS)} markets x 2 groups\n")

# Check 1: rows must add up to the Adobe total of each day
totals = pd.read_csv(os.path.join(RAW_DIR, "totals.csv"), header=None, names=["day"] + metric_cols)
totals = totals.drop_duplicates("day", keep="last")[["day", "revenue"]].rename(columns={"revenue": "revenue_adobe_total"})
checks = pd.DataFrame(checks).merge(totals, how="left", on="day")
checks["diff_pct"] = (checks["revenue_rows"] / checks["revenue_adobe_total"] - 1) * 100
bad = checks[(checks["diff_pct"].abs() > 0.5) | checks["revenue_adobe_total"].isna() | (checks["revenue_low_traffic"] > 0)
             | (checks["revenue_unparsed"] > 0) | (checks["duplicate_keys"] > 0)]
print(f"Check 1: {len(bad)} of {len(checks)} days need a look (total mismatch, low-traffic, unparsed or duplicate keys)")
display(bad)

# Check 2: where the revenue went. PMAX, Unparsed and Other are left out of the final table.
print("\nCheck 2: revenue by market and group, whole period")
display(daily.pivot_table(index="market", columns="campaign_group", values="revenue", aggfunc="sum", fill_value=0).round(0))

# Check 3: accounts the market rule did not recognise
print("\nCheck 3: top accounts in 'Other'")
display(pd.concat(other_accounts).groupby("account", as_index=False)["revenue"].sum()
          .sort_values("revenue", ascending=False).head(15))

# Check 4: does the campaign rule agree with the -Brand account suffix?
print("\nCheck 4: campaign group vs account ending in -Brand (revenue)")
display(pd.concat(brand_check).pivot_table(index="campaign_group", columns="brand_account",
                                           values="revenue", aggfunc="sum", fill_value=0).round(0))

Saved aa_daily_market_group.csv: 1,896 rows = 158 days x 6 markets x 2 groups

Check 1: 17 of 158 days need a look (total mismatch, low-traffic, unparsed or duplicate keys)


,day,rows,duplicate_keys,revenue_rows,revenue_low_traffic,revenue_unparsed,revenue_adobe_total,diff_pct
20,2026-05-21,10942,0,782529.06,0.0,1130.00,782529.06,-1.110223e-14
24,2026-05-25,10065,0,739681.25,0.0,589.65,739681.25,0.000000e+00
30,2026-05-31,9648,0,511454.45,0.0,1915.56,511454.45,0.000000e+00
41,2026-06-11,13056,0,633987.73,0.0,6436.16,633987.73,0.000000e+00
44,2026-06-14,12200,0,616626.49,0.0,173.85,616626.49,0.000000e+00
48,2026-06-18,13726,0,696266.39,0.0,1374.06,696266.39,-2.220446e-14
53,2026-06-23,13553,0,675335.80,0.0,733.47,675335.80,0.000000e+00
67,2026-07-07,14043,0,804918.94,0.0,989.00,804918.94,-1.110223e-14
71,2026-07-11,9793,0,386248.98,0.0,566.00,386248.98,0.000000e+00
110,2026-08-19,13512,0,577962.48,0.0,977.70,577962.48,0.000000e+00



Check 2: revenue by market and group, whole period


campaign_group,Brand,Non-brand,PMAX,Unparsed
market,,,,
Canada,20370255.0,323486.0,1301450.0,0.0
Germany,13431692.0,427457.0,684947.0,0.0
Malaysia,9584804.0,197009.0,309441.0,0.0
Other,63402.0,462433.0,797.0,53596.0
Pakistan,5242087.0,273757.0,0.0,0.0
Saudi Arabia,28731565.0,29100.0,0.0,0.0
Singapore,7405195.0,105842.0,290914.0,0.0



Check 3: top accounts in 'Other'


,account,revenue
126,Google-ME_CAUCASUS-SA-EN-2,410923.96
4,Bing-AMERICAS-US-EN,23293.80
84,Google-Europe-GB-EN-Brand,17139.34
51,Google-AMERICAS-BR-EN-Brand,9838.12
14,Bing-Europe-GB-EN,9633.01
6,Bing-AUASIA_NASIA-AU-EN,7079.66
13,Bing-Europe-FR-FR-Brand,6717.15
56,Google-AMERICAS-US-EN-Brand,4933.90
23,Bing-SEA_SASC-IN-EN,3835.13
128,Google-QATAR-QA-EN-Brand,3742.07



Check 4: campaign group vs account ending in -Brand (revenue)


brand_account,False,True
campaign_group,,
Brand,0.0,84829000.0
Non-brand,1812907.0,6178.0
PMAX,2587549.0,0.0
Unparsed,53596.0,0.0
